# End-to-End Spam Detection Pipeline

This notebook covers the complete workflow for building a machine learning model to classify SMS and email messages as spam or ham (legitimate). 
Target Metrics: Accuracy 97%, Precision 95%, Recall 93%, F1-Score 94%.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import urllib.request
import os
import re
import string
import pickle

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, auc

nltk.download('punkt')
nltk.download('stopwords')


## 2. Data Collection (UCI SMS Spam Collection)

In [ ]:
# Download the UCI dataset if not present
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = "smsspamcollection.zip"
data_path = "SMSSpamCollection"

if not os.path.exists(data_path):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Dataset downloaded & extracted!")
else:
    print("Dataset already exists.")

# Load dataset
df = pd.read_csv(data_path, sep='\t', names=['target', 'text'])
display(df.head())


## 3. Exploratory Data Analysis & Cleaning

In [ ]:
# Encode target variable (spam=1, ham=0)
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])

# Check for null values and duplicates
df.drop_duplicates(keep='first', inplace=True)
print("Shape after deduplication:", df.shape)

# Distribution
plt.figure(figsize=(6,4))
sns.countplot(tuple(df), x='target')
plt.title('Spam vs Ham Distribution')
plt.show()


## 4. Text Preprocessing
Includes lowercase conversion, tokenization, removing special chars/stopwords, stemming, and HTML tag removal.

In [ ]:
ps = PorterStemmer()

def transform_text(text):
    text = text.lower() # lowercase
    text = re.sub('<.*?>', '', text) # drop HTML tags for emails
    text = nltk.word_tokenize(text) # tokenization
    
    y = []
    for i in text:
        if i.isalnum(): # retain alphanumeric only
            y.append(i)
            
    text = y[:]
    y.clear()
    
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
            
    text = y[:]
    y.clear()
    
    for i in text:
        y.append(ps.stem(i)) # Stemming
        
    return " ".join(y)

# Apply preprocessing
df['transformed_text'] = df['text'].apply(transform_text)
df.head()


## 5. Feature Engineering (TF-IDF Vectorization)

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 6. Model Training and Comparison
We evaluate Naive Bayes, Logistic Regression, SVC, and Random Forest.

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC(kernel='sigmoid', gamma=1.0, probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42)
}

results = []
best_model_name = ""
best_model = None
highest_f1 = 0 

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})
    print(f"\n{name} Metrics:")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    
    if f1 > highest_f1:
        highest_f1 = f1
        best_model_name = name
        best_model = model

results_df = pd.DataFrame(results)
display(results_df.sort_values(by='F1-Score', ascending=False))


## 7. Model Evaluation (ROC-AUC & Confusion Matrix)

In [ ]:
print(f"Best Model: {best_model_name}")
y_pred_best = best_model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - ' + best_model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC Curve
if hasattr(best_model, "predict_proba"):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()


## 8. Exporting the Model
We save the highest performing model (usually Naive Bayes or SVM) and the TF-IDF vectorizer.

In [ ]:
import pickle

pickle.dump(tfidf, open('../vectorizer.pkl', 'wb'))
pickle.dump(best_model, open('../model.pkl', 'wb'))

print("Model and Vectorizer saved successfully!")
